# CS431 — Paddle Phase 2 trên Kaggle: gate Phase 1 → review → TextVQA clean/degraded

Notebook này chạy **đúng pipeline repo** `run.py paddle phase1 → paddle review → paddle phase2 → compare`
với **cùng detector như Phase 1 chính thức của Triệu** (`paddle/configs/paddle_v5.json`):
PaddleOCR repo **v3.3.3** (commit `c0522b89…`, không sửa), config `PP-OCRv5_server_det.yml` chính thức,
weights `PP-OCRv5_server_det_pretrained.pdparams` (SHA `2802f7d4…`), `paddlepaddle-gpu 3.3.1` cu126, Python 3.13 của Kaggle.
Tiền xử lý / hậu xử lý y như `tools/eval.py`: `DetResizeForTest` mặc định (cạnh ngắn 736), DBPostProcess 0,3 / 0,6 / 1,5.

Paddle được cài trong **venv riêng** (tránh lỗi torch/NCCL `ncclCommShrink` Triệu gặp khi cài vào môi trường gốc Kaggle).

**T4 và ảnh lớn:** chỉ khi GPU báo OOM, ảnh được chạy lại trên CPU trong tiến trình riêng,
giữ nguyên pixels/resize/checkpoint/ngưỡng/FP32. Tối đa một ảnh CPU cùng lúc. Không bỏ ảnh.
CPU có thể chậm và vẫn cần đủ RAM; nếu CPU lỗi thì dừng. Mỗi prediction ghi thiết bị thực tế.
Kết quả mới ở `outputs/paddle_t4_cpu_v1/`; kết quả cũ được giữ nguyên và không trộn vào lượt mới.
Code đã đổi nên cần chạy lại Phase 1 và người thực hiện review lại.

**Gate Phase 1 trong notebook phải ra lại ≈ P 58,52 / R 47,28 / Hmean 52,30** (cùng code/config/weights/packages với `tools/eval.py`);
`REVIEW_ME.md` in sẵn chênh lệch. Lệch đáng kể = adapter không tái tạo pipeline chính thức → dừng, không review.

### Cài đặt Kaggle
- **Accelerator: GPU T4 ×2**, **Internet: On**.
- **Session options → Persistence: Files only** (nên bật: mất kết nối thì `/kaggle/working` còn lại để resume).
- **Add Input → Datasets** (4 bộ):
  1. ICDAR2015: https://www.kaggle.com/datasets/bestofbests9/icdar2015 (cùng nguồn ảnh với Phase 1 official_eval)
  2. TextVQA clean: https://www.kaggle.com/datasets/anhthu128/text-vqa-train
  3. TextVQA degraded MAIN 10K: https://www.kaggle.com/datasets/anhthu128/textvqa-suy-thoi-10k-nh
  4. TextOCR train JSON (chỉ để audit nguồn, chấm điểm không cần): https://www.kaggle.com/datasets/anhthu128/textocr-0-1-train-json
- Chạy **interactive** (không “Save & Run All”), vì có một bước người thực hiện phải tự xem kết quả Phase 1 rồi điền đánh giá.
  Giữ tab mở trong lúc Phase 2 chạy.

### Thời gian ước tính
Thời gian Phase 2 phụ thuộc số ảnh phải chạy CPU; chưa có thời gian toàn bộ được kiểm chứng.

### Quy tắc (AGENTS.md)
Không tự approve, không giảm số ảnh, không đổi threshold/resize theo test, không rebuild manifest,
không lấy kết quả Monkey hay số 83,8% làm ngưỡng. 52,30 là kiểm tra **tái tạo pipeline** (cùng detector), không phải mục tiêu chất lượng.

## 0. Clone repo đúng commit, kiểm tra GPU

In [ ]:
import os, sys, re, json, time, signal, random, hashlib, shutil, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/ncat218/CS431.git'
REPO_BRANCH = 'paddle-phase2'   # nhánh chứa adapter tools/eval.py-equivalent (hoặc main sau khi merge)
REPO_REF = ''                   # tuỳ chọn: pin một commit SHA cụ thể; để trống = HEAD của REPO_BRANCH (SHA được ghi lại)
BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('/content')
REPO = BASE / 'CS431'
RUN_ROOT = REPO / 'outputs/paddle_t4_cpu_v1'
PHASE1 = RUN_ROOT / 'phase1'
OUT2 = RUN_ROOT / 'phase2'
VENV = BASE / 'cs431-paddle'
PYTHON = str(VENV / 'bin/python')
LOGS = BASE / 'cs431_logs'
LOGS.mkdir(parents=True, exist_ok=True)
CUDA_WHEEL = 'cu126'          # chỉ đổi sang cu118 TRƯỚC Phase 1 nếu driver không hỗ trợ CUDA 12.6
os.environ.pop('PYTHONPATH', None)


def run(cmd, log=None, cwd=None, env=None):
    """Chạy lệnh, in trực tiếp ra notebook và ghi log; lỗi thì dừng."""
    cmd = [str(c) for c in cmd]
    print('$', ' '.join(cmd), flush=True)
    fh = open(log, 'a', encoding='utf-8') if log else None
    try:
        with subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
            for line in p.stdout:
                print(line, end='', flush=True)
                if fh:
                    fh.write(line)
    finally:
        if fh:
            fh.close()
    if p.returncode != 0:
        raise RuntimeError(f'Lệnh lỗi (exit {p.returncode}). Xem log: {log}')


if not (REPO / '.git').is_dir():
    run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, REPO])
elif not REPO_REF:
    branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=REPO, text=True).strip()
    assert branch == REPO_BRANCH, 'Clone thuộc nhánh khác; không tự đổi nhánh.'
    run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO)
if REPO_REF:
    run(['git', 'checkout', '--detach', REPO_REF], cwd=REPO)
HEAD = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert not REPO_REF or HEAD == REPO_REF, HEAD
dirty = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=REPO, text=True).strip()
assert not dirty, 'File tracked trong clone bị sửa:\n' + dirty
assert (REPO / 'cs431/ppocr_detector.py').is_file(), 'Clone chưa có adapter tools/eval.py-equivalent: sai nhánh/commit'
assert (REPO / 'cs431/cpu_fallback.py').is_file(), 'Clone chưa có hỗ trợ CPU khi T4 hết bộ nhớ'
PIN = json.loads((REPO / 'paddle/configs/paddle_v5.json').read_text(encoding='utf-8'))
assert PIN['paddleocr_tag'] == 'v3.3.3' and PIN['package_versions']['paddlepaddle-gpu'] == '3.3.1', PIN
print('Repo:', REPO, '@', HEAD, '| PaddleOCR', PIN['paddleocr_tag'], '| paddle', PIN['package_versions']['paddlepaddle-gpu'])

gpus = subprocess.check_output(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version',
                                '--format=csv,noheader'], text=True).strip().splitlines()
print(*gpus, sep='\n')
GPUS = ','.join(g.split(',')[0].strip() for g in gpus[:2])   # T4 ×2 -> '0,1'; một A100 -> '0'
print('GPU IDs cho workers:', GPUS)
subprocess.run([sys.executable, str(REPO / 'run.py'), 'paddle', 'status'], cwd=REPO)   # clone mới: BLOCKED là đúng

## 1. Môi trường Paddle riêng (venv từ Python 3.13 của Kaggle) + model chính thức
`setup_paddle.py` tạo venv, cài `paddlepaddle-gpu==3.3.1` (cu126) và requirements PaddleOCR v3.3.3 đã pin theo môi trường
của Triệu, chạy `pip check` + GPU matmul probe. `prepare_paddle_model.py` clone PaddleOCR v3.3.3 (kiểm commit, cây sạch,
SHA config, evaluator vendor = upstream) và tải weights `.pdparams` chính thức (kiểm SHA256).
Nếu bước nào lỗi: **dừng**, gửi log trong `/kaggle/working/cs431_logs/`; không tự đổi version.

In [ ]:
t0 = time.time()
print('Kaggle python:', sys.version)
run([sys.executable, REPO / 'paddle/tools/setup_paddle.py', '--venv', VENV, '--cuda', CUDA_WHEEL],
    LOGS / '02_setup_paddle.log', cwd=REPO)
run([PYTHON, REPO / 'paddle/tools/prepare_paddle_model.py'], LOGS / '03_prepare_model.log', cwd=REPO)
run([PYTHON, '-c', 'import importlib.metadata as m, json, sys; '
     'print(sys.version); print(json.dumps({k: m.version(k) for k in ["paddlepaddle-gpu","numpy","opencv-python","shapely","Pillow"]}))'],
    LOGS / '04_versions.log')
print(f'Môi trường xong sau {time.time() - t0:.0f} s')
# Test host RAM on the previously failing image before the full rerun.
probe_images = list(INPUT_ROOT.rglob('21d60e0822064acb.jpg'))
if len(probe_images) != 1:
    raise RuntimeError('Cần tìm đúng ảnh TextVQA từng OOM 21d60e0822064acb.jpg; kiểm tra dataset đã attach.')
run([PYTHON, REPO / 'paddle/tools/probe_cpu_image.py', '--image', probe_images[0],
     '--out', RUN_ROOT / 'cpu_memory_probe'], LOGS / '05_cpu_probe.log', cwd=REPO)

## 2. Phase 1 gate — ICDAR2015 test đủ 500 ảnh
Tự tìm thư mục có **đúng** `img_1 … img_500` và `gt_img_1.txt … gt_img_500.txt` (tập train có 1.000 ảnh nên không khớp).
Có nhiều ứng viên thì dừng và điền tay `ICDAR_IMAGES` / `ICDAR_GT`. Không tải mirror khác.

In [ ]:
ICDAR_IMAGES = ''   # để trống = tự tìm
ICDAR_GT = ''
DATASET_SOURCE = '' # để trống = tự ghi theo dataset bestofbests9/icdar2015 nếu đường dẫn khớp


def dirs_with_exact_ids(root, pattern, n=500):
    want = {str(i) for i in range(1, n + 1)}
    hits = []
    for dp, dn, fn in os.walk(root):
        ids = {m.group(1) for f in fn if (m := re.fullmatch(pattern, f, re.I))}
        if ids == want:
            hits.append(Path(dp))
    return hits


if not ICDAR_IMAGES or not ICDAR_GT:
    img_hits = dirs_with_exact_ids(INPUT_ROOT, r'img_(\d+)\.(?:jpg|jpeg|png)')
    gt_hits = dirs_with_exact_ids(INPUT_ROOT, r'gt_img_(\d+)\.txt')
    print('Ứng viên ảnh:', img_hits)
    print('Ứng viên GT :', gt_hits)
    if len(img_hits) != 1 or len(gt_hits) != 1:
        raise RuntimeError('Cần đúng 1 thư mục ảnh test và 1 thư mục GT test; điền tay ICDAR_IMAGES/ICDAR_GT.')
    ICDAR_IMAGES = ICDAR_IMAGES or str(img_hits[0])
    ICDAR_GT = ICDAR_GT or str(gt_hits[0])
if not DATASET_SOURCE:
    if 'bestofbests9' in ICDAR_IMAGES or 'icdar2015' in ICDAR_IMAGES.lower():
        DATASET_SOURCE = (f'Kaggle https://www.kaggle.com/datasets/bestofbests9/icdar2015 ; images={ICDAR_IMAGES} ; '
                          f'gt={ICDAR_GT} ; cùng nguồn ảnh với paddle/phase1/report/official_eval')
    else:
        raise ValueError('Không nhận ra nguồn ICDAR2015; điền DATASET_SOURCE thật.')
print('ICDAR_IMAGES =', ICDAR_IMAGES)
print('ICDAR_GT     =', ICDAR_GT)
print('SOURCE       =', DATASET_SOURCE)

t0 = time.time()
run([PYTHON, REPO / 'run.py', 'paddle', 'phase1', '--images', ICDAR_IMAGES, '--gt', ICDAR_GT,
     '--dataset-source', DATASET_SOURCE, '--gpus', GPUS, '--out', PHASE1], LOGS / '10_phase1.log', cwd=REPO)
print(f'Phase 1 xong sau {time.time() - t0:.0f} s')

## 3. Xem kết quả Phase 1 trước khi quyết định
Xem P/R/Hmean, counts, log workers, `detector.json` (553/553 tham số, 0 thiếu, 0 lệch shape; resize 736/min) và **20 overlays**
(xanh lá = GT valid, xám = GT ignored `###`, đỏ = prediction).

**Kiểm tra tái tạo:** cùng code/config/weights/packages với `tools/eval.py` của Triệu (P 58,52 / R 47,28 / Hmean 52,30) nên số phải
gần như trùng. Nếu lệch, kiểm tra dữ liệu/runtime/adapter trước khi review; chưa có ngưỡng sai số GPU được xác lập.
Cell dưới chạy thêm CPU trên 3 ảnh cố định để so với GPU và lưu diagnostic bên ngoài report Phase 1.
Nếu hộp chữ khác nhau, dừng để xem xét; mẫu 3 ảnh không chứng minh tương đương trên toàn bộ dữ liệu.
Đây là kiểm tra “cùng detector”, không phải ngưỡng chất lượng; 83,8 vẫn NOT VERIFIED.

In [ ]:
from IPython.display import display, Markdown
from PIL import Image

display(Markdown((PHASE1 / 'REVIEW_ME.md').read_text(encoding='utf-8')))
m1 = json.loads((PHASE1 / 'metrics.json').read_text())
print(json.dumps(m1, indent=2)[:3000])
per = [json.loads(l) for l in (PHASE1 / 'per_image.jsonl').read_text().splitlines() if l.strip()]
print('Ảnh không có prediction nào:', sum(r['predictions'] == 0 for r in per), '/ 500')
REF = PIN['phase1_official_reference']
for k in ('precision', 'recall', 'hmean'):
    print(f'{k:9s} repo {100 * m1["measured"][k]:.3f}%  | tools/eval.py {100 * REF[k]:.3f}%  | Δ {100 * (m1["measured"][k] - REF[k]):+.3f} pp')
print(json.dumps(json.loads(next((PHASE1 / 'inference').glob('gpu*/detector.json')).read_text()), indent=2))
for log in sorted((PHASE1 / 'inference').glob('gpu*.log')):
    print('---', log.name); print('\n'.join(log.read_text(errors='ignore').splitlines()[-5:]))
print(json.dumps(json.loads(next((PHASE1 / 'inference').glob('gpu*/runtime.json')).read_text()), indent=2))
for image in sorted((PHASE1 / 'overlays').glob('*.jpg')):
    print(image.name)
    with Image.open(image) as src:
        preview = src.copy(); preview.thumbnail((1000, 700)); display(preview)
run([PYTHON, REPO / 'paddle/tools/compare_cpu_gpu.py', '--run', PHASE1,
     '--out', RUN_ROOT / 'cpu_comparison'], LOGS / '10_cpu_comparison.log', cwd=REPO)
CPU_COMPARISON = json.loads((RUN_ROOT / 'cpu_comparison/comparison.json').read_text())
print(json.dumps(CPU_COMPARISON, indent=2))

## 4. Người thực hiện ghi nhận đánh giá Phase 1 (Trí tự điền)
Chỉ điền **sau khi đã xem** cell trên. Ghi P/R/Hmean thực tế và chênh lệch so với `tools/eval.py` 52,30, nhận xét overlays
(box theo dòng vs GT theo từ, bỏ sót chữ dọc/xoay…), nguồn dữ liệu và giới hạn (83,8 NOT VERIFIED).
Cell mặc định dừng — đây là quyết định của người thực hiện, không phải bước tự approve.

In [ ]:
REVIEWER = ''
REVIEW_NOTES = ''
if not REVIEWER.strip() or len(REVIEW_NOTES.strip()) < 20:
    raise ValueError('Xem kết quả Phase 1 rồi điền REVIEWER và REVIEW_NOTES (nhận định thực tế) để mở gate Phase 2.')
if not CPU_COMPARISON['all_polygon_sets_equal']:
    raise ValueError('CPU/GPU có hộp chữ khác nhau trong mẫu đối chiếu; cần xem xét trước khi review.')
run([PYTHON, REPO / 'run.py', 'paddle', 'review', '--run', PHASE1, '--reviewer', REVIEWER, '--notes', REVIEW_NOTES],
    LOGS / '11_review.log', cwd=REPO)
run([PYTHON, REPO / 'run.py', 'paddle', 'status'], cwd=REPO)

## 5. Đọc nguồn shared, giải nén manifest đã freeze
Bắt buộc theo AGENTS.md: notebook shared, protocol, data audit, `docs/PHASE2_SHARED_MANIFEST.md`.
Manifest giải nén phải có SHA256 `94e3a2ce…f242` (9.898 cặp). Không join/sample lại.

In [ ]:
public = json.loads((REPO / 'shared/Shared_Manifest_TextVQA_TextOCR_Public.ipynb').read_text())
for cell in public['cells']:
    if cell['cell_type'] == 'markdown':
        display(Markdown(''.join(cell['source'])))
display(Markdown((REPO / 'docs/PHASE2_SHARED_MANIFEST.md').read_text()))
display(Markdown((REPO / 'shared/frozen/data_audit.md').read_text()))
print((REPO / 'shared/frozen/evaluation_protocol.json').read_text())
run([PYTHON, REPO / 'run.py', 'shared', 'unpack'], cwd=REPO)
run([PYTHON, REPO / 'run.py', 'shared', 'verify'], cwd=REPO)
MANIFEST = REPO / 'outputs/shared/shared_manifest_v1.jsonl'
rows = [json.loads(l) for l in MANIFEST.open(encoding='utf-8') if l.strip()]
assert len(rows) == 9898, len(rows)
print('Manifest rows:', len(rows), '| ví dụ:', rows[0]['clean_relpath'], '|', rows[0]['degraded_relpath'])

## 6. Xác định `clean_root` / `degraded_root` từ file thật
Root đúng là thư mục sao cho `root / clean_relpath` (vd `train_images/<id>.jpg`) và `root / degraded_relpath`
(vd `main/images/realistic_mix/L2/<id>.png`) tồn tại. Cell kiểm **toàn bộ 9.898 + 9.898 file** tồn tại và
SHA256 của 50 cặp ngẫu nhiên khớp manifest (workers còn hash lại từng ảnh trước inference).

In [ ]:
CLEAN_ROOT = ''      # để trống = tự tìm
DEGRADED_ROOT = ''


def find_root(relpaths, label, probe=20):
    sample = relpaths[:probe]
    hits = []
    for dp, dn, fn in os.walk(INPUT_ROOT):
        d = Path(dp)
        if all((d / r).is_file() for r in sample[:2]) and all((d / r).is_file() for r in sample):
            hits.append(d)
        if len(d.relative_to(INPUT_ROOT).parts) >= 6:
            dn[:] = []
    print(label, 'ứng viên:', hits)
    if len(hits) != 1:
        raise RuntimeError(f'{label}: cần đúng 1 root, thấy {len(hits)}; điền tay.')
    return str(hits[0])


CLEAN_ROOT = CLEAN_ROOT or find_root([r['clean_relpath'] for r in rows], 'clean_root')
DEGRADED_ROOT = DEGRADED_ROOT or find_root([r['degraded_relpath'] for r in rows], 'degraded_root')
missing = [r['source_image_id'] for r in rows if not (Path(CLEAN_ROOT) / r['clean_relpath']).is_file()
           or not (Path(DEGRADED_ROOT) / r['degraded_relpath']).is_file()]
assert not missing, f'{len(missing)} source thiếu file, ví dụ {missing[:5]}'


def file_sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


for r in random.Random(0).sample(rows, 50):
    assert file_sha(Path(CLEAN_ROOT) / r['clean_relpath']) == r['clean_sha256'], r['source_image_id']
    assert file_sha(Path(DEGRADED_ROOT) / r['degraded_relpath']) == r['degraded_sha256'], r['variant_id']
LOCATIONS = REPO / 'outputs/dataset_locations.json'
LOCATIONS.parent.mkdir(parents=True, exist_ok=True)
LOCATIONS.write_text(json.dumps(dict(clean_root=CLEAN_ROOT, degraded_root=DEGRADED_ROOT), indent=2))
print('OK: đủ 19.796 file, 50 cặp hash khớp. Locations:', LOCATIONS.read_text())

## 7. Phase 2 — inference 2 GPU + chấm shared evaluator + bootstrap + paired overlays
`run.py paddle phase2` kiểm gate trước khi load model, chia task xen kẽ cho 2 worker (mỗi worker thấy 1 GPU),
merge đủ 19.796 `sample_id`, chấm bằng `shared/evaluate_shared.py` (hash `ea25c8d9…`), bootstrap 1.000 lần.
Cell in tiến độ mỗi 60 s. **Bị ngắt?** Chạy lại đúng cell này (cùng `--out`, cùng GPU) — predictions đã ghi được giữ lại.
Nếu báo `Output is locked` sau khi kernel chết: xác nhận controller/workers/CPU child đã dừng trước khi xử lý
`outputs/paddle_t4_cpu_v1/phase2/.controller.lock`. Không xoá file `.cpu_slot.lock` (khóa OS tự nhả).

In [ ]:
cmd = [PYTHON, str(REPO / 'run.py'), 'paddle', 'phase2', '--locations', str(LOCATIONS), '--gpus', GPUS, '--out', str(OUT2)]
print('$', ' '.join(cmd))
t0 = time.time()
log_path = LOGS / '20_phase2.log'
with open(log_path, 'a', encoding='utf-8') as log:
    proc = subprocess.Popen(cmd, cwd=REPO, stdout=log, stderr=subprocess.STDOUT)
    try:
        while proc.poll() is None:
            time.sleep(60)
            parts = []
            for g in sorted((OUT2 / 'inference').glob('gpu*.log')):
                prog = [l for l in g.read_text(errors='ignore').splitlines() if l.startswith('PROGRESS')]
                parts.append(f'{g.stem}: {prog[-1] if prog else "khởi động"}')
            last = log_path.read_text(errors='ignore').splitlines()[-1:] or ['']
            print(f'[{(time.time() - t0) / 60:6.1f} phút]', ' | '.join(parts), '|', last[0][:120], flush=True)
    except KeyboardInterrupt:
        proc.send_signal(signal.SIGINT)   # controller tự dừng workers và gỡ lock
        proc.wait()
        raise
print('\n'.join(log_path.read_text(errors='ignore').splitlines()[-30:]))
if proc.returncode != 0:
    raise RuntimeError(f'Phase 2 lỗi (exit {proc.returncode}); xem {log_path} và {OUT2}/inference/gpu*.log')
print(f'Phase 2 xong sau {(time.time() - t0) / 60:.1f} phút')
display(Markdown((OUT2 / 'metrics/clean_vs_degraded.md').read_text()))

## 8. Bảng so sánh với Monkey (cùng manifest/protocol/evaluator) + phân tích phụ
`run.py compare` từ chối nếu khác hash hoặc khác số source/variant. Latency khác phần cứng → không xếp hạng tốc độ.

In [ ]:
run([PYTHON, REPO / 'run.py', 'compare', '--paddle-metrics', OUT2 / 'metrics/metrics.json'],
    LOGS / '30_compare.log', cwd=REPO)
display(Markdown((REPO / 'comparison/phase2.md').read_text()))

pm = json.loads((OUT2 / 'metrics/metrics.json').read_text())
mm = json.loads((REPO / 'monkey/phase2/report/metrics.json').read_text())
lines = ['| Model | Cond | <16px | 16–32px | ≥32px |', '|---|---|---:|---:|---:|']
for name, m in (('paddle', pm), ('monkey', mm)):
    for cond in ('clean', 'degraded'):
        b = m['size_bins'][cond]
        lines.append(f'| {name} | {cond} | ' + ' | '.join(f'{100 * b[k]["recall"]:.2f}' for k in
                     ('small_lt16px', 'medium_16_32px', 'large_ge32px')) + ' |')
display(Markdown('**Recall theo chiều cao bbox GT gốc (%)**\n\n' + '\n'.join(lines)))
lines = ['| Model | Stratum | Hmean clean | Hmean degraded | Δ (pp) |', '|---|---|---:|---:|---:|']
for name, m in (('paddle', pm), ('monkey', mm)):
    for key, s in m['strata'].items():
        lines.append(f'| {name} | {key} | {100 * s["clean_paired"]["hmean"]:.2f} | '
                     f'{100 * s["degraded"]["hmean"]:.2f} | {s["delta_pp"]["hmean"]:+.2f} |')
display(Markdown('**Theo có/không perspective**\n\n' + '\n'.join(lines)))
preds = [json.loads(l) for l in (OUT2 / 'inference/predictions.jsonl').open() if l.strip()]
for cond in ('clean', 'degraded'):
    sub = [p for p in preds if p['condition'] == cond]
    print(cond, '| ảnh không có prediction:', sum(not p['polygons'] for p in sub), '/', len(sub),
          '| tổng raw polygons:', sum(len(p['polygons']) for p in sub))
print('Gallery overlays:', OUT2 / 'overlays')
print('Thiết bị / thời gian:', (OUT2 / 'inference/timing.json').read_text())

## 9. Đóng gói để review và đưa về repo
Zip gồm: report Phase 1 gate, report Phase 2 (metrics, per-image, paired CSV, predictions, provenance, overlays),
`comparison/phase2.md`, logs, pip freeze của venv. **Tải file zip ở Output** rồi gửi lại để viết `REVIEW.md` Phase 2.
Zip có pixels dataset (overlays) và raw predictions → **không commit nguyên zip lên git**.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
(LOGS / 'pip_freeze_venv.txt').write_text(subprocess.check_output([PYTHON, '-m', 'pip', 'freeze'], text=True))
(LOGS / 'repo_head.txt').write_text(HEAD + '\n')
bundle = BASE / 'CS431_Paddle_Phase2_review.zip'
skip = {'.controller.lock'}
with ZipFile(bundle, 'w', compression=ZIP_DEFLATED) as z:
    for folder, prefix in ((PHASE1, 'phase1_gate'), (OUT2, 'phase2'), (RUN_ROOT / 'cpu_comparison', 'cpu_comparison'),
                           (LOGS, 'logs')):
        for p in sorted(folder.rglob('*')):
            if p.is_file() and p.name not in skip and 'official_eval' not in p.parts and 'diag_line_vs_word' not in p.parts:
                z.write(p, arcname=f'{prefix}/{p.relative_to(folder)}')
    z.write(REPO / 'comparison/phase2.md', arcname='comparison/phase2.md')
h = hashlib.sha256(bundle.read_bytes()).hexdigest()
print('Bundle:', bundle, f'{bundle.stat().st_size / 1e6:.1f} MB', 'sha256', h)